# Decoding What People See From Brain Activity
## Stage 1: Exploring and Cleaning the THINGS-EEG Dataset

**The big idea:** 50 people looked at thousands of pictures of everyday objects (animals, food, tools...) while wearing a cap that records the brain's electrical activity (EEG). In this notebook we download those recordings, understand what's in them, clean out the noise, and cut them into one short clip per picture, so a machine-learning model can later try to guess what each person was looking at.

**Dataset:** THINGS-EEG (OpenNeuro ds003825), Grootswagers et al., 2022, *Scientific Data* 9:3

**How this notebook is organised**
- **Part A (sections 1–11):** we explore and clean **one participant (sub-01) step by step**, to understand the data and choose our cleaning settings.
- **Part B (section 12):** the same steps run **automatically on all 50 participants**.
- **Part C (sections 13–15):** quality results, the final list of usable participants, and how teammates load the clean data.

# Part A: Exploring and cleaning one participant

## 1. Connect to Google Drive
Colab is a temporary computer in the cloud that forgets everything when closed. Connecting Google Drive gives us permanent storage, so our cleaned data is saved safely.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive

## 2. Download the data (one participant)
We install two tools:
- **openneuro-py**: downloads datasets from OpenNeuro, a free public library of brain data
- **MNE**: the standard Python toolbox for working with EEG

We start with **one participant (sub-01)** to learn the data and build our cleaning steps. The full dataset is ~41 GB, so it makes sense to perfect the process on one person first.

**Result:** 738 MB per participant. Each person's folder contains:
- the brain recording itself (`.eeg`, `.vhdr`, `.vmrk` files)
- an events table listing every picture shown and exactly when

*(The red "pip dependency" message during installation is harmless.)*

In [ ]:
!pip install -q openneuro-py mne

import openneuro
openneuro.download(
    dataset="ds003825",
    target_dir="things_eeg",
    include=["sub-01", "participants.tsv", "task-rsvp_events.json", "task-rsvp_eeg.json"],
)

!find things_eeg -maxdepth 3 | sort
!du -sh things_eeg/sub-01

## 3. The labels: what was shown, and when?
The events table is our "answer key": one row for every picture the participant saw.

**Key findings for sub-01:**
- **22,248 pictures** were shown, each for just 50 milliseconds (10 per second)
- **1,854 different objects** (aardvark, airplane, piano...)
- **12 different photos of each object**, each shown once
- Important columns: `onset` (when the picture appeared, in seconds) and `object` (what it was)

In [ ]:
import pandas as pd

events = pd.read_csv("things_eeg/sub-01/eeg/sub-01_task-rsvp_events.tsv", sep="\t")

print("Number of rows:", len(events))
print("Columns:", list(events.columns))
events.head()

## 4. Checking the labels
Quick sanity checks before trusting the labels:
- **1,854 objects**: all present ✅
- **0 broken labels** (object number -1): none for this participant ✅
- **264 "target" pictures**: these were part of a small attention task (participants pressed a button when they spotted them). They aren't normal trials, so we remove them later.

**Why this matters:** 12 examples per object is too few for a model to learn 1,854 separate objects. Objects will later be grouped into broad categories (e.g. animals vs. food) by the feature-engineering stage.

In [ ]:
print("Different objects:", events["object"].nunique())
print("Rows with objectnumber = -1:", (events["objectnumber"] == -1).sum())
print("Target images:", events["istarget"].sum())
print("\nMost common objects:")
print(events["object"].value_counts().head())

## 5. Opening the brain recording
Think of the recording as a giant spreadsheet:
- **63 columns**: one per electrode on the head
- **~3 million rows**: one per millisecond (the machine measured 1,000 times per second)
- **50.6 minutes** long

Electrode names show their position: **F** = front, **C** = centre, **T** = sides, **P** = upper back, **O** = back of the head. The **O and PO electrodes sit over the brain's visual area**, so they matter most for this project.

The top-of-head electrode (**Cz**) is not in the list: it was the "reference" point all other electrodes were measured against.

In [ ]:
import mne

raw = mne.io.read_raw_brainvision(
    "things_eeg/sub-01/eeg/sub-01_task-rsvp_eeg.vhdr", preload=False
)

print("Number of electrodes:", len(raw.ch_names))
print("Electrode names:", raw.ch_names)
print("Samples per second:", raw.info["sfreq"])
print("Recording length (minutes):", round(raw.n_times / raw.info["sfreq"] / 60, 1))

## 6. Filtering: removing noise
Raw EEG contains things that aren't brain activity:
- **Slow drift** (sweat, electrodes settling): removed with a 0.1 Hz high-pass filter
- **Fast noise** (muscles, electronics): removed with a 100 Hz low-pass filter
- **Electrical hum** from power lines (50 Hz in Australia, where it was recorded): removed with a notch filter

We then reduce from 1,000 to 250 measurements per second. That's still plenty of detail, and it makes the data 4x smaller.

**Reading the plots:** each line is one electrode, showing how much of each frequency it contains.
- **Before:** a sharp spike at 50 Hz (power-line hum) and peaks at 10, 20, 30 Hz.
- **After:** the 50 Hz spike is gone ✅. The 10/20 Hz peaks stay on purpose: they are the brain responding to pictures flashing 10 times per second, which is real signal.

In [ ]:
# 1. Load the full recording into memory
raw.load_data()

# Save a copy of the frequency picture BEFORE filtering
fig_before = raw.compute_psd(fmax=150).plot(show=False)
fig_before.suptitle("BEFORE filtering")

# 2. Band-pass filter: keep 0.1 - 100 Hz
raw.filter(l_freq=0.1, h_freq=100)

# 3. Notch filter: remove 50 Hz power-line hum
raw.notch_filter(freqs=50)

# 4. Downsample to 250 samples per second
raw.resample(250)

# 5. Frequency picture AFTER filtering
fig_after = raw.compute_psd(fmax=125).plot(show=False)
fig_after.suptitle("AFTER filtering")

display(fig_before, fig_after)
print("New samples per second:", raw.info["sfreq"])

## 7. Finding bad electrodes
Sometimes an electrode loses contact with the scalp and records only noise or nothing. We compare each electrode to the typical one.

**First attempt:** this flagged forehead electrodes as "noisy" and C1/C2 as "flat". Neither was actually broken:
- Forehead electrodes pick up **eye blinks** (handled later)
- C1/C2 sit right next to the reference point (Cz), so they naturally look quiet

This showed we needed a better check (next cell).

In [ ]:
import numpy as np

data = raw.get_data()                 # all electrodes x all time points
spread = data.std(axis=1)             # how much each electrode varies

# Compare each electrode to the typical electrode
median = np.median(spread)
ratio = spread / median

flat  = [ch for ch, r in zip(raw.ch_names, ratio) if r < 0.2]   # < 20% of typical
noisy = [ch for ch, r in zip(raw.ch_names, ratio) if r > 5]     # > 5x typical

print("Flat electrodes:", flat or "none")
print("Noisy electrodes:", noisy or "none")
print("\nThe 5 most variable electrodes:")
for ch, r in sorted(zip(raw.ch_names, ratio), key=lambda x: -x[1])[:5]:
    print(f"  {ch}: {r:.1f}x typical")

### 7b. A better check
We switch to an **average reference** (comparing each electrode to the average of all electrodes, rather than to Cz) and add a second test that measures **high-frequency noise**, the signature of bad contact or muscle tension.

**Result:** three electrodes behind the ears / at the temple were clearly bad: **TP10 (34x typical noise), TP9 (17x), T8 (14x)**. These sit over jaw and neck muscles, which often cause noise.

**Rule used:** an electrode is bad if its high-frequency noise is more than **10x** the typical electrode.

In [ ]:
# Make a copy using the average of all electrodes as the reference
raw_avg = raw.copy().set_eeg_reference("average", verbose=False)

# Test 1: how much each electrode varies (same as before)
spread = raw_avg.get_data().std(axis=1)
spread_ratio = spread / np.median(spread)

# Test 2: how much high-frequency noise (60-100 Hz) each electrode has
# -> catches electrodes with bad contact or muscle noise
psd = raw_avg.compute_psd(fmin=60, fmax=100, verbose=False)
hf = psd.get_data().mean(axis=1)
hf_ratio = hf / np.median(hf)

check = pd.DataFrame({"electrode": raw.ch_names,
                      "variation_x_typical": spread_ratio.round(1),
                      "high_freq_noise_x_typical": hf_ratio.round(1)})

print("Lowest variation (possibly flat):")
print(check.sort_values("variation_x_typical").head(3).to_string(index=False))
print("\nHighest variation:")
print(check.sort_values("variation_x_typical", ascending=False).head(5).to_string(index=False))
print("\nMost high-frequency noise:")
print(check.sort_values("high_freq_noise_x_typical", ascending=False).head(5).to_string(index=False))

## 8. Repairing bad electrodes
Instead of deleting bad electrodes (which would leave gaps), we **rebuild** each one from its neighbours, like estimating a missing pixel from the pixels around it. This is called **interpolation**.

Steps: load the standard electrode positions → mark TP9, TP10, T8 as bad → switch to average reference → rebuild them from the 60 good electrodes.

*(A pink "FutureWarning" about the montage name is harmless.)*

In [ ]:
# 1. Tell MNE where each electrode sits on the head (standard cap layout)
raw.set_montage("standard_1005")

# 2. Mark the bad electrodes (rule: > 10x typical high-frequency noise)
bad_channels = check.loc[check["high_freq_noise_x_typical"] > 10, "electrode"].tolist()
raw.info["bads"] = bad_channels
print("Marked as bad:", bad_channels)

# 3. Switch to average reference (bad electrodes are left out of the average)
raw.set_eeg_reference("average")

# 4. Rebuild the bad electrodes from their neighbours
raw.interpolate_bads(reset_bads=True)

# 5. Check: plot the frequencies again
fig = raw.compute_psd(fmax=125).plot(show=False)
fig.suptitle("After fixing bad electrodes + average reference")
display(fig)

## 9. Cutting the recording into epochs (one clip per picture)
The 50-minute recording is cut into **one short clip per picture**: from 0.1 s before the picture appeared to 0.6 s after. Each clip becomes one example for the machine-learning model, with its object label attached.

- **Baseline:** the 0.1 s before each picture is used as the "zero level"
- **Target pictures removed** (264)
- **Blink removal:** any clip where an electrode swings more than **150 microvolts** is thrown away. Blinks cause huge swings at the forehead, so this removes them automatically. (We chose this instead of the more complex ICA method because it is simple, automatic and works the same for all 50 participants.)

**Result for sub-01:** 20,468 clean clips kept out of 21,984 (**6.9% removed**). Almost all removals were caused by forehead electrodes, which confirms they were blinks.

**The plot** shows the average brain response over the visual electrodes. Clear bumps ~100–200 ms after the picture prove the clips are aligned with real visual brain activity.

In [ ]:
import numpy as np
import pandas as pd
import mne

SUBJECT   = "sub-01"
TMIN      = -0.1          # window starts 100 ms BEFORE the image
TMAX      = 0.6           # window ends 600 ms AFTER the image
BASELINE  = (-0.1, 0.0)   # the 100 ms before the image = "zero level"
REJECT_UV = 150           # drop a window if any electrode swings > 150 microvolts

# --- 9.1 Load the event table (one row per image shown) ---------------------
events_df = pd.read_csv(f"things_eeg/{SUBJECT}/eeg/{SUBJECT}_task-rsvp_events.tsv", sep="\t")
print(f"Images in event table: {len(events_df):,}")

# --- 9.2 Check the timing column is in seconds -------------------------------
assert np.allclose(events_df["onset"], events_df["sample"] / 1000, atol=0.002), \
    "onset and sample columns disagree - check units before continuing"
print("Timing check passed: 'onset' is in seconds.")

# --- 9.3 Build the event list MNE needs, from the event table ----------------
sfreq = raw.info["sfreq"]                                   # 250 Hz after filtering step
samples = np.round(events_df["onset"].to_numpy() * sfreq).astype(int) + raw.first_samp
events_arr = np.column_stack([samples,
                              np.zeros(len(samples), dtype=int),
                              np.ones(len(samples), dtype=int)])

# --- 9.4 Remove target images (attention task, not normal trials) ------------
keep = (events_df["istarget"] != 1).to_numpy()
events_arr = events_arr[keep]
metadata = events_df.loc[keep].reset_index(drop=True)      # labels travel with epochs
print(f"Removed {(~keep).sum()} target images -> {keep.sum():,} images left")

# --- 9.5 Cut the epochs and reject bad ones ----------------------------------
epochs = mne.Epochs(
    raw, events_arr, event_id={"image": 1},
    tmin=TMIN, tmax=TMAX, baseline=BASELINE,
    metadata=metadata,
    reject=dict(eeg=REJECT_UV * 1e-6),    # MNE works in volts
    preload=True, verbose=False,
)

# --- 9.6 Report ---------------------------------------------------------------
n_before = keep.sum()
n_after = len(epochs)
print(f"\nEpochs kept: {n_after:,} of {n_before:,} "
      f"({100 * (n_before - n_after) / n_before:.1f}% rejected)")
print("Epoch data shape (epochs, electrodes, time points):", epochs.get_data().shape)

reasons = pd.Series([r for log in epochs.drop_log for r in log])
if len(reasons):
    print("\nElectrodes most often responsible for rejection:")
    print(reasons.value_counts().head(8).to_string())

# --- 9.7 Sanity check: average brain response over the visual electrodes ----
evoked = epochs.average()
fig = evoked.plot(picks=["O1", "Oz", "O2", "PO7", "POz", "PO8"], show=False,
                  titles=f"{SUBJECT}: average response to an image (visual electrodes)")
display(fig)

## 10. Timing verification: do the labels match the brain signal?
This is the most important check: if the timing is off, every label would be attached to the wrong brain response.

The recording contains its own time-stamps (markers). We compared them with the events table:

| Marker | Count | Meaning | Match with events table |
|---|---|---|---|
| E 1 | 22,248 | picture appears | within **3 ms** ✅ |
| E 2 | 22,248 | picture disappears | +50 ms (display time) |
| E 3 | 72 | start of each picture stream | n/a |

**Conclusion:** labels and brain signal line up to within 3 milliseconds (one sample at 250 Hz).

In [ ]:
marker_events, marker_ids = mne.events_from_annotations(raw, verbose=False)
tsv_times = events_df["onset"].to_numpy()

print(f"Event table: {len(tsv_times):,} images, first at {tsv_times[0]:.3f} s\n")

for name, code in marker_ids.items():
    t = marker_events[marker_events[:, 2] == code, 0] / raw.info["sfreq"]
    # for every image in the event table, find the closest marker of this type
    idx = np.clip(np.searchsorted(t, tsv_times), 1, len(t) - 1)
    gap_ms = np.minimum(np.abs(t[idx] - tsv_times), np.abs(t[idx - 1] - tsv_times)) * 1000
    print(f"{name}: {len(t):,} markers | first at {t[0]:.3f} s | "
          f"median gap to event table: {np.median(gap_ms):.1f} ms | max gap: {gap_ms.max():.1f} ms")

## 11. Who were the participants?
- **50 participants**, aged **17–30** (average 20.4)
- **36 female, 14 male**
- About half are native English speakers

In [ ]:
participants = pd.read_csv("things_eeg/participants.tsv", sep="\t")

print("Columns:", list(participants.columns))
print("Number of participants:", len(participants))
display(participants.head(10))
display(participants.describe(include="all"))

### 11b. Participants with known problems (researchers' notes)
The researchers left notes on 6 participants:

| Participant | Problem reported | What we did |
|---|---|---|
| sub-01 | Missing a small extra part of the experiment | **Kept**: the main part we use is complete |
| sub-06 | Did not finish the experiment | **Kept**: still 10,444 usable clips |
| sub-18 | Braided hair → poor contact on several electrodes | **Kept**: our checks found it acceptable |
| sub-23 | Screen problems during the experiment | **Excluded**: timing can't be trusted |
| sub-49, sub-50 | Wrong (128-electrode) cap used | sub-50 **kept**; sub-49 **excluded** (no usable data, see section 13) |

In [ ]:
pd.set_option("display.max_colwidth", None)   # show the full note text

flagged = participants[(participants["exclude"] == 1) | participants["note_IZ"].notna()]
display(flagged[["participant_id", "age", "gender", "exclude", "note_IZ"]])

print("Marked exclude = 1:", participants.loc[participants["exclude"] == 1, "participant_id"].tolist())

# Part B: Cleaning all 50 participants

## 12. Running the full cleaning pipeline on all participants
Everything in Part A was done step by step on one person. This cell repeats the **exact same steps automatically** for every participant:

**download → filter → find & repair bad electrodes → average reference → cut into clips → remove blinks → save → delete the raw file → next person**

**Two changes for storage and compatibility:**
- The final data uses a **40 Hz low-pass filter** and **125 measurements per second**, making it 5x smaller (~230 MB per person instead of ~900 MB). This keeps the brainwave ranges most relevant to vision (delta, theta, alpha, beta); gamma is limited to 30–40 Hz.
- For the two participants recorded on a 128-electrode cap, only the standard 63 electrodes are loaded (to save memory), and the missing **FCz** electrode is rebuilt from its neighbours.

**What it produces** (in Google Drive → `things_eeg_clean`):
- `sub-XX_epochs.npz`: cleaned brain clips (clips × 63 electrodes × 88 time points, in microvolts)
- `sub-XX_metadata.csv`: which picture each clip belongs to
- `quality_report.csv`: a summary of problems per participant

**Safe to re-run:** finished participants are skipped automatically. Runtime ≈ 1.5 minutes per participant.

In [ ]:
# ============================================================
# STEP 8 - Run the full preprocessing pipeline on all 50 participants
#
# For each participant:
#   download -> filter -> bad electrodes -> average reference ->
#   epochs -> reject blinks -> save -> delete raw -> next
#
# Output (in your Google Drive, so nothing is lost if Colab disconnects):
#   things_eeg_clean/sub-XX_epochs.npz      EEG epochs (microvolts, float16)
#   things_eeg_clean/sub-XX_metadata.csv    one row per epoch: object, image, etc.
#   things_eeg_clean/quality_report.csv     missing / noisy recordings report
#
# Safe to re-run: participants that are already saved are skipped.
# ============================================================
import gc
import os
import shutil
import traceback

import mne
import numpy as np
import openneuro
import pandas as pd

mne.set_log_level("ERROR")

# ---------------- SETTINGS (write these in your documentation) ----------------
RAW_DIR      = "/content/things_eeg"                            # temporary, local
OUT_DIR      = "/content/drive/MyDrive/things_eeg_clean"        # permanent, Drive
SUBJECTS     = [f"sub-{i:02d}" for i in range(1, 51)]

BANDPASS     = (0.1, 100.0)  # Hz, first filter (needed for bad-electrode check)
NOTCH        = 50.0          # Hz, Australian mains hum
LOWPASS_SAVE = 40.0          # Hz, final low-pass before saving
SFREQ_SAVE   = 125           # Hz, final sampling rate (5x smaller than original)
BAD_HF_RATIO = 10            # electrode is bad if 60-100 Hz noise > 10x typical
FLAT_RATIO   = 0.05          # electrode is flat if variation < 5% of typical
TMIN, TMAX   = -0.1, 0.6     # epoch window (s)
BASELINE     = (-0.1, 0.0)
REJECT_UV    = 150           # drop epoch if any electrode swings > 150 microvolts
MAX_BAD_PCT  = 10            # flag participant if > 10% of electrodes are bad

STANDARD_63 = ['Fp1','Fz','F3','F7','FT9','FC5','FC1','C3','T7','TP9','CP5','CP1',
               'Pz','P3','P7','O1','Oz','O2','P4','P8','TP10','CP6','CP2','C4','T8',
               'FT10','FC6','FC2','F4','F8','Fp2','AF7','AF3','AFz','F1','F5','FT7',
               'FC3','C1','C5','TP7','CP3','P1','P5','PO7','PO3','POz','PO4','PO8',
               'P6','P2','CPz','CP4','TP8','C6','C2','FC4','FT8','F6','AF8','AF4',
               'F2','FCz']


def preprocess(raw, events_df, subject):
    """Clean one participant's recording. Returns (X, metadata, report_dict)."""
    rep = {"subject": subject}

    # --- 1. Electrodes: keep the standard 63 (handles the 128-channel set-ups) ---
    rep["n_channels_in_file"] = len(raw.ch_names)
    raw.pick([ch for ch in STANDARD_63 if ch in raw.ch_names])   # drop extras BEFORE loading
    raw.load_data()                                              # (saves memory on 128-ch files)
    missing = [ch for ch in STANDARD_63 if ch not in raw.ch_names]
    rep["missing_electrodes"] = ",".join(missing)
    if len(missing) > 3:
        raise ValueError(f"too many missing standard electrodes: {missing}")
    if missing:   # e.g. FCz on the 128-channel set-up (used as reference there)
        raw = mne.add_reference_channels(raw, missing)   # add as empty channels
    raw.pick(STANDARD_63)
    rep["recording_min"] = round(raw.n_times / raw.info["sfreq"] / 60, 1)

    # --- 2. Timing check: event table vs image-onset markers (E 1) ---------------
    rep["events_in_table"] = len(events_df)
    rep["onset_markers_E1"] = np.nan
    rep["max_timing_gap_ms"] = np.nan
    try:
        mk_events, mk_ids = mne.events_from_annotations(raw)
        key = [k for k in mk_ids if k.replace(" ", "").endswith("E1")]
        if key:
            t = mk_events[mk_events[:, 2] == mk_ids[key[0]], 0] / raw.info["sfreq"]
            rep["onset_markers_E1"] = len(t)
            tt = events_df["onset"].to_numpy()
            tt = tt[tt <= t[-1] + 0.05]          # only images inside the recording
            idx = np.clip(np.searchsorted(t, tt), 1, len(t) - 1)
            gap = np.minimum(np.abs(t[idx] - tt), np.abs(t[idx - 1] - tt)) * 1000
            rep["max_timing_gap_ms"] = round(float(gap.max()), 1)
    except Exception:
        pass

    # --- 3. Filter + downsample (same as Step 4) ---------------------------------
    raw.filter(*BANDPASS)
    raw.notch_filter(NOTCH)
    raw.resample(250)

    # --- 4. Bad electrodes (same rule as Step 5a) --------------------------------
    raw.set_montage("standard_1005", on_missing="ignore")
    check = raw.copy().set_eeg_reference("average")
    spread = check.get_data().std(axis=1)
    spread_ratio = spread / np.median(spread)
    hf = check.compute_psd(fmin=60, fmax=100).get_data().mean(axis=1)
    hf_ratio = hf / np.median(hf)
    del check
    bads = [ch for ch, h, s in zip(raw.ch_names, hf_ratio, spread_ratio)
            if h > BAD_HF_RATIO or s < FLAT_RATIO]
    bads = sorted(set(bads) | set(missing))     # missing electrodes get rebuilt too
    rep["bad_electrodes"] = ",".join(bads)
    rep["n_bad_electrodes"] = len(bads)
    raw.info["bads"] = bads

    # --- 5. Average reference + repair bad electrodes ----------------------------
    raw.set_eeg_reference("average")
    if bads:
        raw.interpolate_bads(reset_bads=True)

    # --- 6. Final low-pass + downsample for storage ------------------------------
    raw.filter(None, LOWPASS_SAVE)
    raw.resample(SFREQ_SAVE)

    # --- 7. Events: drop target images and unlabelled (-1) objects --------------
    rep["target_images"] = int((events_df["istarget"] == 1).sum())
    rep["unlabelled_minus1"] = int((events_df["objectnumber"] == -1).sum())
    keep = ((events_df["istarget"] != 1) & (events_df["objectnumber"] != -1)).to_numpy()
    meta = events_df.loc[keep].reset_index(drop=True)
    samples = np.round(meta["onset"].to_numpy() * raw.info["sfreq"]).astype(int) + raw.first_samp
    ev = np.column_stack([samples, np.zeros(len(samples), int), np.ones(len(samples), int)])

    # images whose window runs past the end of the recording (e.g. sub-06)
    inside = (samples + int(TMAX * raw.info["sfreq"])) < (raw.first_samp + raw.n_times)
    rep["images_after_recording_end"] = int((~inside).sum())
    ev, meta = ev[inside], meta.loc[inside].reset_index(drop=True)

    # --- 8. Epochs + blink rejection (same as Step 6) ----------------------------
    epochs = mne.Epochs(raw, ev, event_id={"image": 1}, tmin=TMIN, tmax=TMAX,
                        baseline=BASELINE, metadata=meta,
                        reject=dict(eeg=REJECT_UV * 1e-6), preload=True)
    rep["epochs_before_rejection"] = len(meta)
    rep["epochs_kept"] = len(epochs)
    rep["pct_rejected"] = round(100 * (1 - len(epochs) / max(len(meta), 1)), 1)

    X = (epochs.get_data() * 1e6).astype(np.float16)   # volts -> microvolts
    out_meta = epochs.metadata.reset_index(drop=True)
    return X, out_meta, epochs.times, rep


# ---------------- MAIN LOOP ----------------
os.makedirs(OUT_DIR, exist_ok=True)
report_path = os.path.join(OUT_DIR, "quality_report.csv")
report = pd.read_csv(report_path) if os.path.exists(report_path) else pd.DataFrame()

for subject in SUBJECTS:
    out_file = os.path.join(OUT_DIR, f"{subject}_epochs.npz")
    if os.path.exists(out_file):
        print(f"{subject}: already done, skipping")
        continue

    print(f"\n===== {subject} =====")
    try:
        openneuro.download(dataset="ds003825", target_dir=RAW_DIR, include=[subject])
        eeg_dir = os.path.join(RAW_DIR, subject, "eeg")
        raw = mne.io.read_raw_brainvision(
            os.path.join(eeg_dir, f"{subject}_task-rsvp_eeg.vhdr"), preload=False)
        events_df = pd.read_csv(
            os.path.join(eeg_dir, f"{subject}_task-rsvp_events.tsv"), sep="\t")

        X, meta, times, rep = preprocess(raw, events_df, subject)

        np.savez(out_file, X=X, times=times, ch_names=np.array(STANDARD_63))
        meta.to_csv(os.path.join(OUT_DIR, f"{subject}_metadata.csv"), index=False)
        rep["status"] = "ok"
        print(f"  kept {rep['epochs_kept']:,} epochs ({rep['pct_rejected']}% rejected), "
              f"bad electrodes: {rep['bad_electrodes'] or 'none'}")
    except Exception as e:
        rep = {"subject": subject, "status": f"FAILED: {e}"}
        print(f"  FAILED: {e}")
        traceback.print_exc()
    finally:
        shutil.rmtree(os.path.join(RAW_DIR, subject), ignore_errors=True)   # free disk
        for name in ("raw", "X", "meta"):
            globals().pop(name, None)
        gc.collect()

    if len(report):
        report = report[report["subject"] != subject]          # replace old entry if re-run
    report = pd.concat([report, pd.DataFrame([rep])], ignore_index=True)
    report.to_csv(report_path, index=False)   # saved after every participant

print("\nAll done. Report saved to", report_path)

# Part C: Results and hand-over

## 13. Quality results: missing and noisy recordings
**All 50 participants were processed successfully.** Overall:
- **Average 20.4% of clips removed** (mostly blinks; this varies a lot between people)
- **865,682 clean clips** in total

**Problems found:**
- **sub-49**: 0 clips kept. Recorded on the 128-channel cap with only half connected, so the data is pure noise
- **sub-45**: 17 of 63 electrodes bad (27%), too many to rebuild reliably
- **Noisy but usable:** sub-15 (65% of clips removed), sub-16 (53%), sub-19 (51%), sub-13 (45%); sub-06, sub-07, sub-08, sub-17 (10–12 repaired electrodes)

In [ ]:
rep = pd.read_csv("/content/drive/MyDrive/things_eeg_clean/quality_report.csv")

print("Participants:", len(rep), "| status ok:", (rep["status"] == "ok").sum())
print("\nAny failures:")
print(rep.loc[rep["status"] != "ok", ["subject", "status"]].to_string(index=False) or "  none")
print("\nAverage % clips removed:", round(rep["pct_rejected"].mean(), 1))
print("Total clean clips:", int(rep["epochs_kept"].sum()))
print("\nParticipants with the most bad electrodes:")
print(rep.sort_values("n_bad_electrodes", ascending=False)
         [["subject", "n_bad_electrodes", "bad_electrodes", "pct_rejected"]].head(6).to_string(index=False))
print("\nHighest % clips removed:")
print(rep.sort_values("pct_rejected", ascending=False)
         [["subject", "pct_rejected", "epochs_kept"]].head(5).to_string(index=False))

## 14. Final list of usable participants
Simple, documented rules decide who is excluded:
1. **Researcher-reported timing problem** → sub-23
2. **Fewer than 5,000 clean clips** → sub-49
3. **More than 20% of electrodes bad** → sub-45

Everyone else is kept. Participants with >40% clips removed or >9 repaired electrodes are **flagged as noisy** (kept, but worth knowing about).

**Result: 47 usable participants.** The list is saved as `participants_usable.csv` in the shared folder.

In [ ]:
folder = "/content/drive/MyDrive/things_eeg_clean"
rep["exclude"] = False
rep["reason"] = ""

rules = [
    (rep["subject"] == "sub-23",      "monitor problems during recording (researcher note)"),
    (rep["epochs_kept"] < 5000,       "fewer than 5,000 clean clips"),
    (rep["n_bad_electrodes"] > 12,    "more than 20% of electrodes bad"),
]
for mask, why in rules:
    rep.loc[mask, "exclude"] = True
    rep.loc[mask, "reason"] = rep.loc[mask, "reason"].str.cat([why] * mask.sum(), sep="; ").str.strip("; ")

rep["noisy_flag"] = (~rep["exclude"]) & ((rep["pct_rejected"] > 40) | (rep["n_bad_electrodes"] > 9))

rep[["subject", "exclude", "reason", "noisy_flag", "epochs_kept", "pct_rejected", "n_bad_electrodes"]] \
    .to_csv(f"{folder}/participants_usable.csv", index=False)

print("Excluded:"); print(rep.loc[rep["exclude"], ["subject", "reason"]].to_string(index=False))
print("\nUsable participants:", (~rep["exclude"]).sum(), "of 50")
print("Clean clips in usable participants:", int(rep.loc[~rep["exclude"], "epochs_kept"].sum()))
print("Flagged as noisy (kept):", rep.loc[rep["noisy_flag"], "subject"].tolist())

## 15. Figures for the presentation
Three figures, saved into the shared folder:
1. **Bad electrodes on a head map** (sub-01): shows *where* the problems were (behind the ears)
2. **Average brain response to a picture** (sub-01): proof the cleaned data contains real visual brain activity
3. **Data quality per participant**: % of clips removed and number of repaired electrodes (red = flagged by the researchers)

In [ ]:
import matplotlib.pyplot as plt

# ---------- Figure 1: Where were the bad electrodes? (sub-01) ----------
z = np.load(f"{folder}/sub-01_epochs.npz")
ch_names = list(z["ch_names"])
info = mne.create_info(ch_names, 125, "eeg")
info.set_montage("standard_1005", on_missing="ignore")
info["bads"] = ["TP9", "TP10", "T8"]          # found in section 7b
fig1 = info.plot_sensors(show_names=True, show=False)
fig1.suptitle("sub-01: electrode layout (bad electrodes in red)")
fig1.savefig(f"{folder}/figure_bad_electrodes.png", dpi=200, bbox_inches="tight")
display(fig1)

# ---------- Figure 2: Average brain response to a picture (sub-01) ----------
X, times = z["X"].astype(np.float32), z["times"]
fig2, ax = plt.subplots(figsize=(8, 4))
for name in ["O1", "Oz", "O2", "PO7", "POz", "PO8"]:
    ax.plot(times * 1000, X[:, ch_names.index(name), :].mean(axis=0), label=name)
ax.axvline(0, color="k", ls="--", lw=1)
ax.axhline(0, color="grey", lw=0.5)
ax.axvspan(0, 50, color="gold", alpha=0.25, label="picture on screen")
ax.set_xlabel("Time after picture appears (ms)")
ax.set_ylabel("Voltage (µV)")
ax.set_title(f"sub-01: average brain response over visual electrodes ({len(X):,} pictures)")
ax.legend(ncol=4, fontsize=8)
fig2.savefig(f"{folder}/figure_average_response.png", dpi=200, bbox_inches="tight")
display(fig2)

# ---------- Figure 3: Data quality across participants ----------
r = rep[rep["status"] == "ok"].sort_values("subject")
flagged_ids = {"sub-01", "sub-06", "sub-18", "sub-23", "sub-49", "sub-50"}
colors = ["tab:red" if s in flagged_ids else "tab:blue" for s in r["subject"]]
fig3, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.bar(r["subject"], r["pct_rejected"], color=colors)
a1.set_ylabel("% clips removed\n(blinks / noise)")
a1.set_title("Data quality per participant (red = flagged by the researchers)")
a2.bar(r["subject"], r["n_bad_electrodes"], color=colors)
a2.set_ylabel("Bad electrodes\n(repaired)")
a2.tick_params(axis="x", rotation=90)
fig3.savefig(f"{folder}/figure_quality.png", dpi=200, bbox_inches="tight")
display(fig3)

## 16. Hand-over: how teammates load the clean data
The clean data is in the shared Google Drive folder **`things_eeg_clean`**. Teammates do **not** need to re-run this notebook; they just load the files.

This cell creates **one combined labels file** for everyone (`ALL_metadata.csv`) and a small helper, `load_eeg([...])`, that loads the brain clips for any participants you choose. Use `participants_usable.csv` to pick only the usable ones.

**Notes for the next stage:**
- One clip per image, −0.1 to 0.6 s, values in µV, 125 Hz, 63 electrodes
- Low-passed at 40 Hz → gamma limited to 30–40 Hz
- Target images and blink clips already removed
- The `object` column = what was shown (link to the 27 THINGS categories for class labels)

In [ ]:
import glob, os

# 1. One combined labels file for everyone
parts = []
for f in sorted(glob.glob(f"{folder}/sub-*_metadata.csv")):
    m = pd.read_csv(f)
    m.insert(0, "subject", os.path.basename(f)[:6])
    parts.append(m)
all_meta = pd.concat(parts, ignore_index=True)
all_meta.to_csv(f"{folder}/ALL_metadata.csv", index=False)
print(all_meta.shape, "->", all_meta["subject"].nunique(), "participants")

# 2. Loader: get the brain data for any participants you choose
def load_eeg(subjects):
    X, meta = [], []
    for s in subjects:
        X.append(np.load(f"{folder}/{s}_epochs.npz")["X"])
        meta.append(all_meta[all_meta["subject"] == s])
    return np.concatenate(X), pd.concat(meta, ignore_index=True)

# example: load 3 participants together
usable = pd.read_csv(f"{folder}/participants_usable.csv")
usable_ids = usable.loc[~usable["exclude"], "subject"].tolist()
X, meta = load_eeg(usable_ids[:3])
print(X.shape, len(meta))